# The Price is Right: Used Car Bargain Hunter

## Retrieval-Augmented Generation (RAG) setup for Used Cars

This notebook focuses strictly on setting up the Chroma vector database with the Hugging Face (HF) dataset.

In [ ]:
# imports

import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm
from agents.items import Item

In [ ]:
# environment

load_dotenv(override=True)
DB = "products_vectorstore"

In [ ]:
# Log in to Hugging Face (HF)
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print("Warning: No HF_TOKEN found in environment variables.")

In [ ]:
# Load the used car dataset using the updated Item class
train, val, test = Item.from_hub("Carson-Shively/used-car-price")
print(f"Loaded {len(train):,} training cars, {len(val):,} validation cars, {len(test):,} test cars")

In [ ]:
# Set up Chroma Vector Database and SentenceTransformer
client = chromadb.PersistentClient(path=DB)
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
collection_name = "used_cars"
existing_collection_names = [collection.name for collection in client.list_collections()]

In [ ]:
if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    print("Populating the vector store. This may take a few minutes...")
    
    for i in tqdm(range(0, len(train), 1000)):
        batch = train[i: i+1000]
        documents = [item.summary for item in batch]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in batch]
        ids = [f"doc_{j}" for j in range(i, i+len(batch))]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)
        
    print("Done!")
else:
    print(f"Collection '{collection_name}' already exists.")
    collection = client.get_collection(collection_name)

In [ ]:
# Test the FrontierAgent manually
from agents.frontier_agent import FrontierAgent
agent = FrontierAgent(collection)
estimated_price = agent.price("2019 Toyota RAV4 LE with 40000 miles. Good condition used car.")
print(f"Estimated Price: ${estimated_price}")